# Capstone Setup & Project Overview (Week 1–4)

**Project:** Design, Evaluation, and Deployment of DNA and Protein Language Model Pipelines for Biological Sequence Analysis  
**Student:** Deepika Sarala Pratapa  
**Goal:** Build end-to-end applied ML pipelines using pretrained DNA/protein language models and compare them against traditional sequence features, with a Streamlit app for decision support and interpretability.

This notebook is the “entry point” for the project:
- Defines the two locked tasks (DNA + Protein)
- Defines data sources (public biological databases)
- Confirms the local environment is correct and reproducible
- Loads the project configuration used across notebooks

## Locked Scope 

**DNA task:** Regulatory element classification (promoter vs non-promoter or enhancer vs non-enhancer)  
**Protein task:** Protein family / function classification

**Constraints:**
- Use existing pretrained DNA/protein language models (no new model invention)
- Use public biological databases (UniProt, Ensembl/RefSeq)
- Applied, reproducible, Master’s-level pipeline
- Final deliverable includes a working Streamlit application
- Keep scope tight and controlled

## What will be built (Weeks 1–4 focus)

**Week 1–2:** Data ingestion + dataset creation + EDA  
- Protein: UniProt (reviewed) + Pfam labels → processed dataset + plots  
- DNA: Ensembl regulatory annotations + reference genome → processed dataset + plots  

**Week 3:** Traditional baselines  
- DNA: k-mer + GC/CpG features → baseline classifier  
- Protein: amino acid composition + basic physicochemical features → baseline classifier  

**Week 4:** Frozen LM embeddings + simple classifiers  
- Protein: ESM-2 embeddings → classifier + comparison vs baseline  
- DNA: DNABERT-2 or Nucleotide Transformer embeddings → classifier + comparison vs baseline  

Artifacts saved for later deployment:
- processed CSVs
- trained models
- (cached) embeddings
- metrics JSON + plots

In [1]:
from pathlib import Path
import os
import sys
import platform
import json
import random
import numpy as np
import pandas as pd
import yaml

# Project paths (this notebook should be in /notebooks)
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
CONFIGS = ROOT / "configs"

for p in [RAW, INTERIM, PROCESSED, MODELS, REPORTS, FIGURES, CONFIGS]:
    p.mkdir(parents=True, exist_ok=True)

ROOT

PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone')

In [2]:
cfg_path = CONFIGS / "config.yaml"
assert cfg_path.exists(), f"Missing config file: {cfg_path}"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg

{'project': {'random_seed': 42},
 'dna': {'organism': 'human',
  'genome_build': 'GRCh38',
  'seq_length_bp': 200,
  'n_pos': 2000,
  'n_neg': 2000},
 'protein': {'organism': 'human',
  'max_len_aa': 1024,
  'n_families': 10,
  'per_family': 400}}

In [3]:
SEED = int(cfg["project"]["random_seed"])

random.seed(SEED)
np.random.seed(SEED)

print("Random seed set to:", SEED)

Random seed set to: 42


In [4]:
import sklearn
import torch
import transformers
from Bio import SeqIO

env_info = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "cwd": str(Path.cwd()),
    "root": str(ROOT),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}

env_info

/opt/anaconda3/envs/bioseq-capstone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python_version': '3.10.19',
 'platform': 'macOS-26.2-arm64-arm-64bit',
 'cwd': '/Users/saturnine/Projects/bio-seq-lm-capstone/notebooks',
 'root': '/Users/saturnine/Projects/bio-seq-lm-capstone',
 'numpy': '2.2.5',
 'pandas': '2.3.3',
 'sklearn': '1.7.2',
 'torch': '2.10.0',
 'transformers': '5.1.0'}

## Data sources (public databases)

**Protein data**
- Source: UniProtKB (reviewed / Swiss-Prot)
- Labels: Pfam family IDs (via UniProt cross-references)
- Output (saved locally): `data/processed/protein_*.csv`

**DNA data**
- Source: Ensembl regulatory annotations + reference genome FASTA
- Labels: promoter vs non-promoter (prototype)
- Output (saved locally): `data/processed/dna_*.csv`

**Data management note**
Raw downloads and large artifacts (embeddings/models) are stored locally under `data/` and `models/` and are not committed to GitHub.

In [5]:
run_log = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "seed": SEED,
    "config": cfg,
    "env": env_info,
}

log_path = REPORTS / "run_log_week1_4_setup.json"
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)

log_path

PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/reports/run_log_week1_4_setup.json')

## Next

Proceed to:
- `01_protein_ingest_eda.ipynb` (UniProt + Pfam ingestion and EDA)